In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

In [ ]:

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
# HYPER PARAMETERS

batch_size = 128
num_epochs = 100
random_seed = 42
test_size = 0.2
learning_rate = 1e-3
total_features = 723

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df_image = pd.read_csv('/content/drive/MyDrive/Adobe/im_embeds_all300k.csv')
df_rest = pd.read_csv('/content/drive/MyDrive/Adobe/Final_input_without_Img_embedding.csv')
df_errors = pd.read_csv('/content/drive/MyDrive/Adobe/im_embeds_errors_all300k.csv')
to_be_deleted = list(df_errors['0'])
df_rest.drop(to_be_deleted,inplace=True)
df_rest.reset_index(drop=True,inplace=True)
df_rest_temp = df_rest.dropna(axis=1, how='any')
df_rest_temp

,January,February,March,April,May,June,July,August,September,October,...,27.8,28.6,28.8,29.6,29.8,30.6,30.8,31.6,31.8,likes
0,0,0,0,0,0,0,0,0,0,0,...,-0.007212,1.014997,0.035538,-1.545867,-0.000829,-0.691397,-0.024089,-0.831907,-0.010203,1
1,0,0,0,0,0,1,0,0,0,0,...,-0.013411,-0.059734,0.008625,-0.008493,0.048716,-0.001403,-0.037191,0.017013,-0.004296,2750
2,0,0,0,0,0,0,0,0,1,0,...,0.018361,0.128244,0.034831,-0.035135,0.046096,0.001906,-0.037285,0.001735,-0.002121,57
3,0,0,0,0,0,0,0,0,0,1,...,0.013616,-0.347948,-0.046577,0.228142,0.036993,-0.067138,0.011416,-0.093506,0.004011,152
4,0,0,0,0,0,0,0,0,0,1,...,0.021757,-0.059734,-0.009485,-0.008493,0.027996,-0.001403,0.015712,0.017013,-0.005038,41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295568,0,0,0,0,0,0,0,0,1,0,...,0.017732,0.032348,0.004959,-0.038293,-0.032024,0.002040,0.015522,-0.011169,0.060232,0
295569,0,1,0,0,0,0,0,0,0,0,...,-0.012335,0.156163,0.058726,0.033617,-0.038409,-0.049654,-0.014553,-0.040862,0.014289,46
295570,0,0,0,0,0,0,0,0,0,0,...,0.027571,-0.271534,0.045534,0.075950,0.013581,-0.061556,-0.011162,0.003064,-0.038035,261
295571,0,0,0,0,0,0,0,0,0,1,...,0.008562,0.032348,0.024369,-0.038293,0.003715,0.002040,-0.016958,-0.011169,0.015464,119


In [ ]:
df_image.drop(['Unnamed: 0'], axis=1, inplace=True)
df_image

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
0,0.126831,0.491699,-0.048615,0.266113,0.446289,-0.708008,0.560547,0.472412,0.166748,0.595703,...,-0.296387,0.231689,-0.171997,0.044220,-0.131592,0.441895,0.123047,0.318604,-0.005260,-0.028809
1,-0.002289,0.313721,-0.461426,0.191406,-0.361572,0.117859,-0.204712,0.558594,-0.088806,0.498535,...,0.138306,-0.169189,0.043549,0.015587,-0.220947,-0.436035,-0.275879,0.244141,-0.082642,0.199097
2,-0.028671,-0.220215,-0.000583,-0.280518,0.505371,-0.867188,-0.128540,0.026566,-0.115967,0.082458,...,0.376953,0.444092,-0.296387,-0.458252,-0.251709,-0.353760,0.401123,0.008011,-0.091980,0.050354
3,0.179565,-0.107056,0.148193,-0.283447,-0.042694,-0.725586,0.185913,0.205322,0.394775,0.139282,...,0.015266,-0.092834,-0.345947,-0.076782,-0.614746,-0.084290,-0.310059,0.819336,-0.333984,0.205688
4,-0.211670,-0.013672,-0.369629,-0.050629,-0.299805,0.185303,0.349365,0.100647,-0.259766,0.023956,...,-0.241211,0.214478,-0.356445,-0.498535,-0.416260,0.172729,0.078918,0.054016,-0.134277,0.002558
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295568,-0.364746,0.036713,0.124329,0.181519,-0.013016,-0.166382,-0.428711,0.222656,-0.069580,0.093140,...,0.284180,0.185791,-0.216797,0.378906,0.322510,0.132935,0.027740,0.480957,-0.399414,-0.461670
295569,-0.003925,-0.021545,-0.414551,-0.294678,0.399170,-0.449463,-0.042847,-0.303467,0.266602,0.155151,...,-0.003538,-0.124695,0.557617,0.248169,0.341309,0.172729,-0.551270,-0.302490,0.114197,-0.663574
295570,-0.284180,-0.304932,0.454102,0.420166,-0.345947,0.016678,-0.088013,0.318359,0.381104,0.115540,...,0.129883,0.418701,-0.133301,-0.275146,-0.284912,0.099365,-0.489990,-0.013573,-0.043396,0.127441
295571,0.547852,0.017517,-0.515137,-0.114746,-0.091736,0.159302,0.063171,-0.116089,0.117432,0.078979,...,-0.260498,0.113098,-0.755859,-0.347412,0.301025,0.044342,-0.117737,-0.496826,0.135498,0.154663


In [ ]:
df_final = pd.concat([df_rest_temp,df_image],axis=1)
df_final.shape

(295573, 724)

In [ ]:
likes = df_final['likes']
likes

0            1
1         2750
2           57
3          152
4           41
          ... 
295568       0
295569      46
295570     261
295571     119
295572     714
Name: likes, Length: 295573, dtype: int64

In [ ]:
max = likes.max()
min = likes.min()
print(max, min)

560193 0


In [ ]:
# NORMALIZATION

numerical_cols = df_final.select_dtypes(include=['number']).columns.drop('likes').tolist()
non_numerical_cols = [col for col in df_final.columns if col not in numerical_cols and col != 'likes']

'''numerical_cols = df_final.select_dtypes(include=['number']).columns
non_numerical_cols = [col for col in df_final.columns if col not in numerical_cols]'''

# Create a copy of the DataFrame to avoid modifying the original
scaled_df = df_final.copy()

# Min-max scaling for numerical columns excluding 'likes'
scaler = MinMaxScaler()
scaled_df[numerical_cols] = scaler.fit_transform(df_final[numerical_cols])

scaled_df

In [ ]:
scaled_df['likes']

In [ ]:
# DATA LOADER

features = scaled_df.drop('likes', axis=1)
targets = scaled_df['likes']

train_features, val_features, train_targets, val_targets = train_test_split(
    features, targets, test_size=test_size, random_state=random_seed
)

class TabularDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features.values, dtype=torch.float32)
        self.targets = torch.tensor(targets.values, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

train_dataset = TabularDataset(train_features, train_targets)
val_dataset = TabularDataset(val_features, val_targets)

train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle=True)
val_loader = DataLoader(val_dataset,
                        batch_size=batch_size,
                        shuffle=False)

In [ ]:
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, total_features):
        super(Model, self).__init__()

        self.fc1 = nn.Linear(total_features, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(1024, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.4)

        self.fc3 = nn.Linear(512, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.4)

        '''self.fc4 = nn.Linear(256, 128)
        self.bn4 = nn.BatchNorm1d(128)
        self.relu4 = nn.ReLU()
        self.dropout4 = nn.Dropout(0.4)  '''

        self.fc5 = nn.Linear(128, 32)
        self.bn5 = nn.BatchNorm1d(32)
        self.relu5 = nn.ReLU()
        self.dropout5 = nn.Dropout(0.4)

        self.fc6 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.dropout3(x)

        '''x = self.fc4(x)
        x = self.bn4(x)
        x = self.relu4(x)
        x = self.dropout4(x)'''

        x = self.fc5(x)
        x = self.bn5(x)
        x = self.relu5(x)
        x = self.dropout5(x)

        output = self.fc6(x)

        return output


In [ ]:
model = Model(total_features=total_features).to(device)

optimizer = torch.optim.AdamW(params=model.parameters(), lr=learning_rate, weight_decay=0.001)
loss_function = nn.MSELoss()

train_loss = []
val_loss = []

In [ ]:
from torchsummary import summary
summary(model,(723,))

In [ ]:
import matplotlib.pyplot as plt

train_losses = []
val_losses = []
for e in tqdm(range(num_epochs)):
    model.train()

    steps_train = 0
    steps_val = 0
    tot_train_loss = 0
    tot_val_loss = 0

    for (x, y) in train_loader:
        (x, y) = (x.to(device), y.to(device))
        pred = model(x).reshape(-1)
        #print(f'pred: {pred.shape}')
        #print(y.shape)
        loss = loss_function(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tot_train_loss += loss.item()
        steps_train += 1

    train_loss_per_epoch = tot_train_loss / steps_train
    train_losses.append(train_loss_per_epoch)

    with torch.no_grad():
        model.eval()

        for (x, y) in val_loader:
            (x, y) = (x.to(device), y.to(device))
            pred = model(x).reshape(-1)
            batch_loss = loss_function(pred, y)
            tot_val_loss += batch_loss.item()  # Use item() to get the scalar value
            steps_val += 1


    val_loss_per_epoch = tot_val_loss / steps_val
    val_losses.append(val_loss_per_epoch)

    print(f"\nEpoch {e + 1} => Train Loss: {train_loss_per_epoch}, Val Loss: {val_loss_per_epoch}")

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()